In [ ]:
%run ../shared_notebook_setup.py
import mlflow

# Specify the tracking server URI, e.g. http://localhost:5000
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("openai-api-basic-chat-observability")
print("MLflow experiment set: openai-api-basic-chat-observability")

MLflow experiment set: openai-api-basic-chat-observability


In [25]:
@mlflow.trace(name="chat_completion", span_type="LLM")
def _traced_chat_completion(messages, temperature):
    return client.chat.completions.create(
        model=DATABRICKS_ENDPOINT,
        messages=messages,
        temperature=temperature,
    )


def chat_once(user_message, history=None, temperature=0.2):
    """Send one chat request and log observability data to MLflow runs + traces."""
    history = history or []
    messages = history + [{"role": "user", "content": user_message}]

    with mlflow.start_run(run_name="openai_chat_turn") as run:
        response = _traced_chat_completion(messages=messages, temperature=temperature)

        assistant_text = response.choices[0].message.content or ""
        usage = response.usage

        mlflow.log_param("model", DATABRICKS_ENDPOINT)
        mlflow.log_param("temperature", temperature)
        mlflow.log_param("input_chars", len(user_message))
        mlflow.log_metric("prompt_tokens", usage.prompt_tokens if usage else 0)
        mlflow.log_metric("completion_tokens", usage.completion_tokens if usage else 0)
        mlflow.log_metric("total_tokens", usage.total_tokens if usage else 0)

        mlflow.log_text(user_message, "input_prompt.txt")
        mlflow.log_text(assistant_text, "assistant_response.txt")
        mlflow.set_tag("component", "chat")
        mlflow.set_tag("status", "success")

        print(f"MLflow run logged: {run.info.run_id}")
        print(f"MLflow trace logged: {mlflow.get_last_active_trace_id()}")

    updated_history = messages + [{"role": "assistant", "content": assistant_text}]
    return assistant_text, updated_history


In [28]:
chat_history = []
assistant_reply, chat_history = chat_once(
    "Give me a two-line intro to MLflow observability.",
    history=chat_history,
    temperature=0.2,
 )
print("Assistant:", assistant_reply)

exp = mlflow.get_experiment_by_name("openai-api-basic-chat-observability")
if exp:
    traces = mlflow.search_traces(
        experiment_ids=[exp.experiment_id],
        max_results=5,
    )
    print("Recent traces:")
    display(traces)
else:
    print("Experiment not found.")

MLflow run logged: 1a0709a40c694cb5890c97e4accd0a37
MLflow trace logged: tr-3fe437b93d704ea56e8fe29eb0fef280
🏃 View run openai_chat_turn at: http://127.0.0.1:5000/#/experiments/1/runs/1a0709a40c694cb5890c97e4accd0a37
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Assistant: Here is a two-line intro to MLflow observability:

MLflow observability enables data scientists and ML engineers to monitor and understand the performance of their machine learning models in production, allowing them to identify issues and opportunities for improvement. By tracking metrics, logs, and other relevant data, MLflow observability provides a comprehensive view of model behavior and helps teams optimize their models for better outcomes.
Recent traces:


C:\Users\akhawaja\AppData\Local\Temp\ipykernel_83552\4108987649.py:11: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(


,trace_id,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-e06dfc0c98e81cc3e9b432efb40171b3,"{""info"": {""trace_id"": ""tr-e06dfc0c98e81cc3e9b4...",None,OK,1781120902223,2338,"{'messages': [{'role': 'user', 'content': 'Giv...",{'id': 'chatcmpl_a30af332-9d71-4cb6-828c-dd7f4...,"{'mlflow.user': 'akhawaja', 'mlflow.source.git...","{'mlflow.traceName': 'chat_completion', 'mlflo...","[{'trace_id': '4G38DJjoHMPptDLvtAFxsw==', 'spa...",[]
1,tr-9c734b80f73e740817642e646f274137,"{""info"": {""trace_id"": ""tr-9c734b80f73e74081764...",None,OK,1781120883587,2250,"{'messages': [{'role': 'user', 'content': 'Giv...",{'id': 'chatcmpl_3748aedd-3d68-4304-8c09-33a64...,"{'mlflow.user': 'akhawaja', 'mlflow.source.git...","{'mlflow.traceName': 'chat_completion', 'mlflo...","[{'trace_id': 'nHNLgPc+dAgXZC5kbydBNw==', 'spa...",[]


Trace(trace_id=tr-e06dfc0c98e81cc3e9b432efb40171b3)

In [32]:
assistant_reply, chat_history = chat_once("Now summarize that in one sentence.", history=chat_history)
print("Assistant:", assistant_reply)

MLflow run logged: a3e4b37e7fa5459ebece92c75b6a0416
MLflow trace logged: tr-d411eee1a37a8345bbfe8cb15c5cfb0e
🏃 View run openai_chat_turn at: http://127.0.0.1:5000/#/experiments/1/runs/a3e4b37e7fa5459ebece92c75b6a0416
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Assistant: MLflow observability helps teams monitor machine learning models.


[Trace(trace_id=tr-d2c20bf6f37ee38ef1123201317a3369), Trace(trace_id=tr-f86d9019820b993ab600a0e984815fc4)]